In [1]:
from __future__ import annotations
import os
import gc
import sys
import copy
import math
import json
import random
import warnings
import logging
import itertools
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (
    GINEConv,
    JumpingKnowledge,
    global_add_pool,
    global_mean_pool,
)

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.metrics import roc_auc_score, average_precision_score

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

/Users/albina/anaconda3/envs/project_ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("DEVICE = mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("DEVICE = cuda")
else:
    DEVICE = torch.device("cpu")
    print("DEVICE = cpu")

CPU = torch.device("cpu")


def set_seed(seed: int) -> None:
    """Make a single experimental cell reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _empty_cache() -> None:
    """Release backend allocator memory.  Called after every seed."""
    if DEVICE.type == "mps":
        try:
            torch.mps.empty_cache()
        except AttributeError:
            pass
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()


DEVICE = mps


In [ ]:
@dataclass
class Config:
    # --- experimental grid ----------------------------------------------------
    datasets:      tuple = ("tox21", "bbbp")
    splits:        tuple = ("scaffold", "mw")
    architectures: tuple = ("gin_jk", "dmpnn", "pretrained_gin")
    uq_methods:    tuple = ("baseline", "mcdropout", "ensemble")
    seeds:         tuple = (0, 1, 2)

    # --- training -------------------------------------------------------------
    hidden_dim:    int   = 128
    num_layers:    int   = 3
    dropout:       float = 0.5
    batch_size:    int   = 64
    lr:            float = 1e-3
    weight_decay:  float = 1e-5
    epochs:        int   = 100
    patience:      int   = 15

    # --- uncertainty ----------------------------------------------------------
    T_mc:            int   = 50              # MC-Dropout passes
    M_ensemble:      int   = 5               # ensemble size
    n_bins:          int   = 15              # ECE / reliability bins
    conformal_alpha: float = 0.1             # 90 % nominal coverage
    n_bootstrap:     int   = 200             # bootstrap draws per bin

    # --- splits ---------------------------------------------------------------
    frac_train:    float = 0.8
    frac_val:      float = 0.1
    frac_test:     float = 0.1

    # --- pretrained checkpoint (point 4) --------------------------------------
    contextpred_url:  str = (
        "http://snap.stanford.edu/gnn-molecular-prediction/model_gin/contextpred.pth"
    )
    contextpred_file: str = "contextpred.pth"
    pretrained_hidden: int = 300
    pretrained_layers: int = 5

    # --- i/o ------------------------------------------------------------------
    out_dir:  str = "./results"
    fig_dir:  str = "./figures"
    data_dir: str = "./data"


CFG = Config()
for d in (CFG.out_dir, CFG.fig_dir, CFG.data_dir):
    os.makedirs(d, exist_ok=True)


# Tox21 task list.
TOX21_TASKS: tuple = (
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
)
DATASET_TASKS = {
    "tox21": TOX21_TASKS,
    "bbbp":  ("p_np",),
}
DATASET_FILES = {
    "tox21": ("tox21.csv.gz", "smiles"),
    "bbbp":  ("BBBP.csv",     "smiles"),
}

In [ ]:
# =============================================================================
# 3.  Data fetch
# =============================================================================
def _fetch_inputs() -> None:
    import urllib.request
    files = {
        "tox21.csv.gz": "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz",
        "BBBP.csv":     "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv",
        CFG.contextpred_file: CFG.contextpred_url,
    }
    for name, url in files.items():
        path = os.path.join(CFG.data_dir, name)
        if os.path.exists(path):
            print(f"[fetch]  {name}  already present")
            continue
        try:
            print(f"[fetch]  downloading {name} ...")
            urllib.request.urlretrieve(url, path)
        except Exception as e:
            print(f"[fetch]  could not download {name}: {e}")

In [ ]:
# =============================================================================
# 4.  Featurisation  (atom + bond)
# =============================================================================
ATOM_LIST = list(range(1, 119))
HYBRID = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
    Chem.rdchem.HybridizationType.UNSPECIFIED,
]
CHIRAL = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
    Chem.rdchem.ChiralType.CHI_OTHER,
]
BOND_TYPE = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

# Vocabulary sizes, with one extra "unknown" bucket per categorical feature.
ATOM_DIMS = [
    len(ATOM_LIST) + 1,   # atomic number (incl. unknown)
    len(CHIRAL)    + 1,   # chirality
    11,                   # degree (0..10)
    11,                   # formal charge + 5  (-5..5)
    5,                    # num radical electrons  (0..4)
    len(HYBRID)    + 1,   # hybridisation
    2,                    # is aromatic
    2,                    # is in ring
    9,                    # total Hs (0..8)
]
BOND_DIMS = [len(BOND_TYPE) + 1, 2, 2]


def _safe_index(lst, val) -> int:
    try:
        return lst.index(val)
    except ValueError:
        return len(lst)            # 'unknown' bucket


def atom_features(atom) -> list[int]:
    return [
        _safe_index(ATOM_LIST, atom.GetAtomicNum()),
        _safe_index(CHIRAL,    atom.GetChiralTag()),
        min(atom.GetDegree(), 10),
        min(max(atom.GetFormalCharge() + 5, 0), 10),
        min(atom.GetNumRadicalElectrons(), 4),
        _safe_index(HYBRID, atom.GetHybridization()),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing()),
        min(atom.GetTotalNumHs(), 8),
    ]


def bond_features(bond) -> list[int]:
    return [
        _safe_index(BOND_TYPE, bond.GetBondType()),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
    ]


def mol_to_pyg(mol: Chem.Mol, y: np.ndarray) -> Optional[Data]:
    """Turn an RDKit Mol into a PyG Data object with .long edge_index."""
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.long)

    # Bonds are stored as PAIRED directed edges:  (i,j) immediately followed
    # by (j,i).  This pairing is essential for _reverse_edge_index.
    ei, ea = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        feat = bond_features(b)
        ei += [[i, j], [j, i]]
        ea += [feat, feat]
    if ei:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.long)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, 3), dtype=torch.long)

    y_t = torch.tensor(y, dtype=torch.float).unsqueeze(0)   # [1, T]
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y_t)

In [ ]:
# =============================================================================
# 5.  Dataset loading
# =============================================================================
def load_dataset(name: str) -> tuple[list[Data], list[str], np.ndarray]:
    """Return (data_list, smiles_list, label_matrix Y[N,T] with NaNs preserved).

    Tox21 uses ONLY the 12 labels listed in TOX21_TASKS.  Any
    extra columns in the CSV (Mol weights, sample IDs, etc.) are ignored.
    NaNs in Y mark missing labels and are preserved for masked loss/metrics.
    """
    fname, smi_col = DATASET_FILES[name]
    df = pd.read_csv(os.path.join(CFG.data_dir, fname))
    tasks = list(DATASET_TASKS[name])

    for c in tasks:
        if c not in df.columns:
            raise KeyError(f"{name}: expected task column '{c}' is missing.")

    Y  = df[tasks].to_numpy(dtype=float)        # NaNs preserved
    sm = df[smi_col].astype(str).tolist()

    data_list, smiles_ok, keep_idx = [], [], []
    for i, s in enumerate(sm):
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            continue
        d = mol_to_pyg(mol, Y[i])
        if d is None:
            continue
        data_list.append(d)
        smiles_ok.append(s)
        keep_idx.append(i)

    Y = Y[keep_idx]
    print(
        f"[{name}]  molecules: {len(data_list):5d}  |  tasks: {len(tasks):2d}  "
        f"|  label density: {np.mean(~np.isnan(Y)):.3f}"
    )
    return data_list, smiles_ok, Y


In [8]:
# =============================================================================
# 6.  Splits  (Scaffold + Molecular Weight -- point 5)
# =============================================================================
def _scaffold(smi: str) -> str:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)


def scaffold_split(smiles: Sequence[str], seed: int):
    """Greedy Bemis-Murcko scaffold split.  Seed only tie-breaks equal-size
    buckets so the train/test partition is essentially deterministic."""
    buckets: dict[str, list[int]] = defaultdict(list)
    for i, s in enumerate(smiles):
        buckets[_scaffold(s)].append(i)
    rng = np.random.default_rng(seed)
    bucket_list = list(buckets.values())
    bucket_list.sort(key=lambda g: (len(g), rng.random()), reverse=True)
    n = len(smiles)
    n_train = int(CFG.frac_train * n)
    n_val   = int(CFG.frac_val   * n)
    train, val, test = [], [], []
    for grp in bucket_list:
        if len(train) + len(grp) <= n_train:
            train.extend(grp)
        elif len(val) + len(grp) <= n_val:
            val.extend(grp)
        else:
            test.extend(grp)
    return train, val, test


def mw_split(smiles: Sequence[str], seed: int):
    """Molecular-Weight split: heaviest molecules go to test (size-OOD)."""
    mw = np.array([
        Descriptors.MolWt(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else np.nan
        for s in smiles
    ])
    order   = np.argsort(mw)                    # light -> heavy
    n       = len(smiles)
    n_train = int(CFG.frac_train * n)
    n_val   = int(CFG.frac_val   * n)
    train = order[:n_train].tolist()
    val   = order[n_train:n_train + n_val].tolist()
    test  = order[n_train + n_val:].tolist()    # heaviest = OOD
    rng = np.random.default_rng(seed)
    rng.shuffle(train)
    rng.shuffle(val)
    return train, val, test


SPLIT_FN = {"scaffold": scaffold_split, "mw": mw_split}

In [24]:
# =============================================================================
# 7.  Embedding helpers + reverse-edge map
# =============================================================================
class AtomEmbedding(nn.Module):
    """Sum of per-feature lookup tables.  Output shape: [N, hidden]."""

    def __init__(self, hidden: int):
        super().__init__()
        self.embs = nn.ModuleList(
            [nn.Embedding(d, hidden) for d in ATOM_DIMS]
        )
        for emb in self.embs:
            nn.init.xavier_uniform_(emb.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [N, num_features], dtype=long
        out = self.embs[0](x[:, 0])
        for i in range(1, len(self.embs)):
            xi = x[:, i].clamp(0, self.embs[i].num_embeddings - 1)
            out = out + self.embs[i](xi)
        return out


class BondEmbedding(nn.Module):
    """Sum of per-feature lookup tables for bond features.  Output: [E, hidden]."""

    def __init__(self, hidden: int):
        super().__init__()
        self.embs = nn.ModuleList(
            [nn.Embedding(d, hidden) for d in BOND_DIMS]
        )
        for emb in self.embs:
            nn.init.xavier_uniform_(emb.weight)

    def forward(self, ea: torch.Tensor) -> torch.Tensor:
        # ea: [E, num_bond_features], dtype=long
        if ea.numel() == 0:
            return torch.zeros((0, self.embs[0].embedding_dim),
                               device=ea.device, dtype=torch.float)
        out = self.embs[0](ea[:, 0])
        for i in range(1, len(self.embs)):
            xi = ea[:, i].clamp(0, self.embs[i].num_embeddings - 1)
            out = out + self.embs[i](xi)
        return out


def _reverse_edge_index(edge_index: torch.Tensor) -> torch.Tensor:
    """Per-edge index of the reverse edge (assumes adjacent-pair layout)."""
    n_edges = edge_index.size(1)
    if n_edges == 0:
        return torch.zeros((0,), dtype=torch.long, device=edge_index.device)
    rev = torch.empty(n_edges, dtype=torch.long, device=edge_index.device)
    rev[0::2] = torch.arange(1, n_edges, 2, device=edge_index.device)
    rev[1::2] = torch.arange(0, n_edges, 2, device=edge_index.device)
    return rev

In [ ]:
# =============================================================================
# 8.  MPS sparse-bypass: keep graph-conv submodules on CPU permanently.
# =============================================================================
#
def to_device(model: nn.Module, device: torch.device) -> nn.Module:
    """Move model to `device` but keep graph-conv submodules pinned to CPU."""
    model.to(device)
    model._device = device
    for m in getattr(model, "_cpu_modules", []):
        m.to(CPU)
    return model

In [ ]:
# =============================================================================
# 9.  Models  (point 4: GIN-JK, D-MPNN, Pretrained-GIN)
# =============================================================================
class GIN_JK(nn.Module):
    """GIN with edge features (GINEConv) + Jumping Knowledge concat readout."""

    def __init__(self, num_tasks: int, hidden: int = 128,
                 n_layers: int = 3, dropout: float = 0.5):
        super().__init__()
        self._device = CPU                           # set by to_device()
        self.atom_emb = AtomEmbedding(hidden)
        self.bond_emb = BondEmbedding(hidden)

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden, 2 * hidden),
                nn.ReLU(),
                nn.Linear(2 * hidden, hidden),
            )
            self.convs.append(GINEConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden))
            self.dropouts.append(nn.Dropout(dropout))

        self.jk = JumpingKnowledge(mode="cat")
        self.head = nn.Sequential(
            nn.Linear(hidden * n_layers, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_tasks),
        )

        # MPS bypass: convs (incl. their internal MLPs) live on CPU.
        self._cpu_modules = list(self.convs)

    def forward(self, data: Data) -> torch.Tensor:
        # Atom / bond embeddings run on DEVICE.
        h  = self.atom_emb(data.x.to(self._device))
        ea = self.bond_emb(data.edge_attr.to(self._device))
        ei = data.edge_index.long()

        xs = []
        for conv, bn, dp in zip(self.convs, self.bns, self.dropouts):
            h_new = conv(h.to(CPU), ei.to(CPU), ea.to(CPU))
            h = h_new.to(self._device)
            h = bn(h)
            h = F.relu(h)
            h = dp(h)
            xs.append(h)

        cat = self.jk(xs)
        pooled_cpu = global_add_pool(cat.to(CPU), data.batch.to(CPU).long())
        pooled = pooled_cpu.to(self._device)
        return self.head(pooled)


class _DMPNNCore(nn.Module):

    def __init__(self, hidden: int, n_steps: int, dropout: float):
        super().__init__()
        self.n_steps = n_steps
        self.W_i = nn.Linear(2 * hidden, hidden, bias=False)
        self.W_h = nn.Linear(hidden, hidden, bias=False)
        self.W_o = nn.Linear(2 * hidden, hidden)
        self.dropout = nn.Dropout(dropout)


    def forward(
        self,
        x: torch.Tensor,           # [N, H]   atom hidden states  (CPU)
        e: torch.Tensor,           # [E, H]   bond hidden states  (CPU)
        edge_index: torch.Tensor,  # [2, E]   long edge index     (CPU)
    ) -> torch.Tensor:
        src, dst = edge_index[0], edge_index[1]

        # ---- Initial bond hidden state h_e^(0)
        h_init = F.relu(self.W_i(torch.cat([x[src], e], dim=-1)))

        # Reverse-edge index uses the (i,j),(j,i) adjacent-pair convention
        # from mol_to_pyg(), so edges 2k and 2k+1 are reverses of each other.
        rev = _reverse_edge_index(edge_index)

        # ---- Iterative message passing --------------------------------------
        h_e = h_init
        for _ in range(self.n_steps):
            # Aggregate incoming edge messages at each node.
            m_v = torch.zeros_like(x)
            m_v.index_add_(0, dst, h_e)
            # D-MPNN reverse-edge subtraction:
            #   m_e[uv] = (sum over (k,u) of h_ku) - h_vu
            m_e = m_v[src] - h_e[rev]
            h_e = F.relu(h_init + self.W_h(m_e))
            h_e = self.dropout(h_e)

        # ---- Final per-atom readout -----------------------------------------
        m_v = torch.zeros_like(x)
        m_v.index_add_(0, dst, h_e)
        h_v = F.relu(self.W_o(torch.cat([x, m_v], dim=-1)))
        return self.dropout(h_v)


class DMPNN(nn.Module):

    def __init__(self, num_tasks: int, hidden: int = 128,
                 n_layers: int = 3, dropout: float = 0.5):
        super().__init__()
        self._device = CPU
        self.atom_emb = AtomEmbedding(hidden)
        self.bond_emb = BondEmbedding(hidden)
        self.core = _DMPNNCore(hidden=hidden, n_steps=n_layers, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_tasks),
        )
        self._cpu_modules = [self.core]

    def forward(self, data: Data) -> torch.Tensor:
        x  = self.atom_emb(data.x.to(self._device))
        e  = self.bond_emb(data.edge_attr.to(self._device))
        ei = data.edge_index.long()
        h_v_cpu = self.core(x.to(CPU), e.to(CPU), ei.to(CPU))
        pooled_cpu = global_mean_pool(h_v_cpu, data.batch.to(CPU).long())
        pooled = pooled_cpu.to(self._device)
        return self.head(pooled)


class PretrainedGIN(nn.Module):
    """5-layer GIN with hidden=300
    everything else is randomly initialised.  This is the honest, defensible compromise --
    the MoleculeNet pretraining used a 2-feature atom encoder, while we use
    a 9-feature one, so a perfect state_dict match is impossible without
    re-training the encoder.
    """

    def __init__(self, num_tasks: int,
                 hidden: int = 300, n_layers: int = 5,
                 dropout: float = 0.5):
        super().__init__()
        self._device = CPU
        self.atom_emb = AtomEmbedding(hidden)
        self.bond_emb = BondEmbedding(hidden)
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden, 2 * hidden),
                nn.ReLU(),
                nn.Linear(2 * hidden, hidden),
            )
            self.convs.append(GINEConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden))
            self.dropouts.append(nn.Dropout(dropout))
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_tasks),
        )
        self._cpu_modules = list(self.convs)

        # ---- attempt to load pretrained weights ------------------------------
        # contextpred.pth uses different key names and a 2-feature atom encoder (vs. our 9-feature one), so a strict load is impossible.  We remap whatever we can:
        #   gnns.{i}.eps                ->  convs.{i}.eps
        #   gnns.{i}.mlp.{j}.weight/.bias  ->  convs.{i}.nn.{j}.weight/.bias
        #   batch_norms.{i}.{w,b,running_mean,running_var,num_batches_tracked}
        #                               ->  bns.{i}.<same>
        #   x_embedding{1,2}.weight     ->  atom_emb.embs.{0,1}.weight
        #
        # Everything else (head, bond embedding, atom features 2..8) stays at random-init
        path = os.path.join(CFG.data_dir, CFG.contextpred_file)
        if os.path.isfile(path):
            try:
                sd = torch.load(path, map_location="cpu", weights_only=False)
                if isinstance(sd, dict) and "state_dict" in sd:
                    sd = sd["state_dict"]

                local_sd = self.state_dict()
                matched: dict[str, torch.Tensor] = {}

                def _try(remote: str, local: str) -> bool:
                    if (remote in sd and local in local_sd
                            and sd[remote].shape == local_sd[local].shape):
                        matched[local] = sd[remote]
                        return True
                    return False

                for i in range(n_layers):
                    _try(f"gnns.{i}.eps",          f"convs.{i}.eps")
                    for j in (0, 2):
                        _try(f"gnns.{i}.mlp.{j}.weight",
                             f"convs.{i}.nn.{j}.weight")
                        _try(f"gnns.{i}.mlp.{j}.bias",
                             f"convs.{i}.nn.{j}.bias")
                    for tag in ("weight", "bias", "running_mean",
                                "running_var", "num_batches_tracked"):
                        _try(f"batch_norms.{i}.{tag}", f"bns.{i}.{tag}")

                # Atom-feature embeddings (only first 2 features overlap)
                _try("x_embedding1.weight", "atom_emb.embs.0.weight")
                _try("x_embedding2.weight", "atom_emb.embs.1.weight")

                local_sd.update(matched)
                self.load_state_dict(local_sd, strict=False)
                print(
                    f"[pretrained_gin]  loaded {len(matched)}/{len(local_sd)} "
                    f"tensors from {path}  "
                    f"({len(local_sd) - len(matched)} stayed random-init)"
                )
            except Exception as e:
                print(f"[pretrained_gin]  could not load {path}: {e}  -- "
                      f"falling back to random init")
        else:
            print(f"[pretrained_gin]  {path} not present  -- using random init")

    def forward(self, data: Data) -> torch.Tensor:
        h  = self.atom_emb(data.x.to(self._device))
        ea = self.bond_emb(data.edge_attr.to(self._device))
        ei = data.edge_index.long()
        for conv, bn, dp in zip(self.convs, self.bns, self.dropouts):
            h_new = conv(h.to(CPU), ei.to(CPU), ea.to(CPU))
            h = h_new.to(self._device)
            h = bn(h)
            h = F.relu(h)
            h = dp(h)
        pooled_cpu = global_mean_pool(h.to(CPU), data.batch.to(CPU).long())
        pooled = pooled_cpu.to(self._device)
        return self.head(pooled)


MODEL_REGISTRY = {
    "gin_jk":         GIN_JK,
    "dmpnn":          DMPNN,
    "pretrained_gin": PretrainedGIN,
}


def build_model(arch: str, num_tasks: int) -> nn.Module:
    cls = MODEL_REGISTRY[arch]
    if arch == "pretrained_gin":
        return cls(
            num_tasks=num_tasks,
            hidden=CFG.pretrained_hidden,
            n_layers=CFG.pretrained_layers,
            dropout=CFG.dropout,
        )
    return cls(
        num_tasks=num_tasks,
        hidden=CFG.hidden_dim,
        n_layers=CFG.num_layers,
        dropout=CFG.dropout,
    )

In [12]:
# =============================================================================
# 10.  Loss  (NaN-masked, macro-averaged across tasks  --  point 2)
# =============================================================================
class MaskedBCEWithLogitsLoss(nn.Module):
    """BCE-with-logits that ignores NaN positions and averages PER TASK
    before averaging across tasks, so dense tasks cannot dominate sparse ones.
    """

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        mask   = ~torch.isnan(targets)
        target = torch.where(mask, targets, torch.zeros_like(targets))
        loss   = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
        loss   = loss * mask.float()
        denom  = mask.float().sum(dim=0).clamp_min(1.0)        # per-task valid count
        return (loss.sum(dim=0) / denom).mean()                # macro across tasks


In [ ]:
# =============================================================================
# 11.  Training loop
# =============================================================================
def make_loaders(data_list, train_idx, val_idx, test_idx):
    pin = (DEVICE.type == "cuda")
    kw  = dict(batch_size=CFG.batch_size, pin_memory=pin, num_workers=0)
    tr = DataLoader([data_list[i] for i in train_idx], shuffle=True,  **kw)
    va = DataLoader([data_list[i] for i in val_idx],   shuffle=False, **kw)
    te = DataLoader([data_list[i] for i in test_idx],  shuffle=False, **kw)
    return tr, va, te


def train_one_model(model: nn.Module, train_loader: DataLoader,
                    val_loader: DataLoader, seed: int) -> nn.Module:
    set_seed(seed)
    model = to_device(model, DEVICE)       
    opt = torch.optim.Adam(model.parameters(),
                           lr=CFG.lr, weight_decay=CFG.weight_decay)
    loss_fn = MaskedBCEWithLogitsLoss()

    best_val, best_state, bad = float("inf"), None, 0
    for _ in range(CFG.epochs):
        # ---- train ----------------------------------------------------------
        model.train()
        for batch in train_loader:
            # Move y to DEVICE; the model's forward handles per-tensor moves
            # for x / edge_index / edge_attr so the CPU bypass works correctly.
            y = batch.y.to(model._device)
            opt.zero_grad(set_to_none=True)
            logits = model(batch)
            loss   = loss_fn(logits, y)
            loss.backward()
            opt.step()

        # ---- validate -------------------------------------------------------
        model.eval()
        vloss, nb = 0.0, 0
        with torch.inference_mode():
            for batch in val_loader:
                y = batch.y.to(model._device)
                vloss += loss_fn(model(batch), y).item()
                nb += 1
        vloss /= max(nb, 1)

        # ---- early stopping -------------------------------------------------
        if vloss < best_val - 1e-5:
            best_val = vloss
            best_state = copy.deepcopy(model.state_dict())
            bad = 0
        else:
            bad += 1
            if bad > CFG.patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        for m in getattr(model, "_cpu_modules", []):
            m.to(CPU)

    return model



In [ ]:
# =============================================================================
# 12.  Uncertainty inference  (deterministic / MC-Dropout / ensemble)
# =============================================================================
def enable_dropout_only(model: nn.Module) -> int:
    """MC-Dropout activation.

    1.  model.eval()  -> freezes BatchNorm running stats.
    2.  Re-enable .train() ONLY on nn.Dropout* modules so dropout masks are
        re-sampled on every forward pass.
    Returns the count of re-activated dropout modules so the caller can
    assert that the model is, in fact, stochastic.
    """
    model.eval()
    n = 0
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout1d,
                          nn.Dropout2d, nn.Dropout3d, nn.AlphaDropout)):
            m.train()
            n += 1
    return n


@torch.inference_mode()
def predict_deterministic(model: nn.Module, loader: DataLoader
                          ) -> tuple[np.ndarray, np.ndarray]:
    """One forward pass per batch.  Returns (probs[N,T], labels[N,T])."""
    model.eval()
    P, Y = [], []
    for batch in loader:
        p = torch.sigmoid(model(batch)).detach().cpu().numpy()
        P.append(p)
        Y.append(batch.y.detach().cpu().numpy())
    return np.concatenate(P), np.concatenate(Y)


@torch.inference_mode()
def predict_mcdropout(model: nn.Module, loader: DataLoader,
                      T: int = None) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """T stochastic passes with dropout-only enabled.  Returns
    (mean prob, variance, labels)."""
    if T is None:
        T = CFG.T_mc
    n_dp = enable_dropout_only(model)
    assert n_dp > 0
    per_batch, ys = [], []
    for batch in loader:
        pT = []
        for _ in range(T):
            pT.append(torch.sigmoid(model(batch)))
        stacked = torch.stack(pT, dim=0).detach().cpu().numpy()    # [T, B, k]
        per_batch.append(stacked)
        ys.append(batch.y.detach().cpu().numpy())
    probs_T = np.concatenate(per_batch, axis=1)                    # [T, N, k]
    return probs_T.mean(axis=0), probs_T.var(axis=0), np.concatenate(ys)


def predict_ensemble(models: list[nn.Module], loader: DataLoader
                     ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Sigmoid-average across M independently-trained members."""
    probs_m, y_ref = [], None
    for m in models:
        p, y = predict_deterministic(m, loader)
        probs_m.append(p)
        y_ref = y
    probs = np.stack(probs_m, axis=0)                              # [M, N, k]
    return probs.mean(axis=0), probs.var(axis=0), y_ref



In [ ]:
# =============================================================================
# 13.  Metrics  (point 7: AURC /AECE / Brier / NLL / Conformal)
# =============================================================================
def _valid(y: np.ndarray, p: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Drop positions where y is NaN (per-task NaN masking)."""
    m = ~np.isnan(y)
    return y[m], p[m]


def mcp_confidence(p: np.ndarray) -> np.ndarray:
    """Maximum Class Probability:  max(p, 1-p).
    Used for the deterministic baseline so it can be plotted on the same
    Risk-Coverage axes as the UQ methods."""
    return np.maximum(p, 1.0 - p)


def ece_binary(probs: np.ndarray, labels: np.ndarray,
               n_bins: int = None) -> float:
    """Equal-WIDTH ECE:  weighted abs(acc - conf) over n_bins bins."""
    n_bins = n_bins or CFG.n_bins
    conf = mcp_confidence(probs)
    pred = (probs >= 0.5).astype(int)
    acc  = (pred == labels).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece, N = 0.0, len(probs)
    for i in range(n_bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1])
        if m.sum() == 0:
            continue
        ece += (m.sum() / N) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def adaptive_ece_binary(probs: np.ndarray, labels: np.ndarray,
                        n_bins: int = None) -> float:
    """Equal-MASS (adaptive) ECE"""
    n_bins = n_bins or CFG.n_bins
    conf = mcp_confidence(probs)
    pred = (probs >= 0.5).astype(int)
    acc  = (pred == labels).astype(float)
    order = np.argsort(conf)
    conf_s, acc_s = conf[order], acc[order]
    bins = np.array_split(np.arange(len(conf_s)), n_bins)
    ece, N = 0.0, len(conf_s)
    for b in bins:
        if len(b) == 0:
            continue
        ece += (len(b) / N) * abs(acc_s[b].mean() - conf_s[b].mean())
    return float(ece)


def brier_binary(probs: np.ndarray, labels: np.ndarray) -> float:
    return float(np.mean((probs - labels) ** 2))


def nll_binary(probs: np.ndarray, labels: np.ndarray, eps: float = 1e-7) -> float:
    p = np.clip(probs, eps, 1 - eps)
    return float(-np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p)))


def auc_err(variance: np.ndarray, probs: np.ndarray, labels: np.ndarray
            ) -> float:
    """AUROC of variance for predicting the binary error indicator.
    Returns NaN if variance is constant (deterministic baseline) or the
    error vector is degenerate (all 0 or all 1)."""
    pred = (probs >= 0.5).astype(int)
    err  = (pred != labels).astype(int)
    if err.sum() == 0 or err.sum() == len(err) or np.var(variance) < 1e-12:
        return float("nan")
    return float(roc_auc_score(err, variance))


def risk_coverage_aurc(
    confidence: np.ndarray,
    probs: np.ndarray,
    labels: np.ndarray,
):

    confidence = np.asarray(confidence, dtype=float).ravel()
    probs      = np.asarray(probs,      dtype=float).ravel()
    labels     = np.asarray(labels,     dtype=float).ravel()
    mask = np.isfinite(confidence) & np.isfinite(probs) & np.isfinite(labels)
    confidence, probs, labels = confidence[mask], probs[mask], labels[mask]
    n = len(probs)
    if n == 0:
        return np.array([]), np.array([]), float("nan")

    order = np.argsort(-confidence, kind="stable")
    p_s, y_s = probs[order], labels[order]
    brier    = (p_s - y_s) ** 2
    risk     = np.cumsum(brier) / np.arange(1, n + 1)
    coverage = np.arange(1, n + 1) / n

    # ---- FULL [0, 1] DOMAIN: prepend (0, risk[0]) ---------------------------
    coverage_aug = np.concatenate([[0.0], coverage])
    risk_aug     = np.concatenate([[risk[0]], risk])

    _trapz = getattr(np, "trapezoid", None) or np.trapz
    aurc = float(_trapz(risk_aug, coverage_aug))
    return coverage, risk, aurc


def split_conformal_binary(cal_p: np.ndarray, cal_y: np.ndarray,
                           test_p: np.ndarray, test_y: np.ndarray,
                           alpha: float = None) -> tuple[float, float]:
    """Split conformal prediction for binary classification.

    Non-conformity:    s_i = 1 - P(true class).
    Set at level 1-a:  C(x) = { c : P(c) >= 1 - qhat }.

    Returns
    -------
    coverage : float        empirical fraction of test_y inside its set
    mean size: float        average |C(x)| over the test split  (in {0,1,2})
    """
    if alpha is None:
        alpha = CFG.conformal_alpha
    cal_y_int = cal_y.astype(int)
    p2 = np.stack([1.0 - cal_p, cal_p], axis=1)
    s_cal = 1.0 - p2[np.arange(len(cal_p)), cal_y_int]
    q_level = np.ceil((len(cal_p) + 1) * (1 - alpha)) / len(cal_p)
    q_level = float(np.clip(q_level, 0.0, 1.0))
    qhat = np.quantile(s_cal, q_level, method="higher")

    t2 = np.stack([1.0 - test_p, test_p], axis=1)
    sets = (1.0 - t2) <= qhat
    set_sizes = sets.sum(axis=1)
    test_y_int = test_y.astype(int)
    covered = sets[np.arange(len(test_p)), test_y_int]
    return float(covered.mean()), float(set_sizes.mean())


def per_task_metrics(probs: np.ndarray,
                     variance: Optional[np.ndarray],
                     labels: np.ndarray,
                     cal_probs: Optional[np.ndarray] = None,
                     cal_labels: Optional[np.ndarray] = None
                     ) -> tuple[dict, pd.DataFrame]:
    """Compute every metric per task, then macro-average across tasks.

    The aggregate dict stores macro-mean values and is the one that gets
    macro-averaged AGAIN across seeds in the final summary.  The per-task
    DataFrame is exported as the Tox21 appendix.
    """
    T_tasks = labels.shape[1]
    rows = []
    for k in range(T_tasks):
        y_k, p_k = _valid(labels[:, k], probs[:, k])
        v_k = (variance[:, k][~np.isnan(labels[:, k])]
               if variance is not None else np.zeros_like(p_k))

        if len(y_k) < 10 or y_k.sum() == 0 or y_k.sum() == len(y_k):
            rows.append(dict(
                task=k, roc_auc=np.nan, pr_auc=np.nan,
                ece=np.nan, aece=np.nan, brier=np.nan, nll=np.nan,
                auc_err=np.nan, aurc=np.nan,
                conf_cov=np.nan, conf_size=np.nan,
                n=len(y_k),
            ))
            continue

        try:
            roc = roc_auc_score(y_k, p_k)
        except Exception:
            roc = np.nan
        try:
            pr = average_precision_score(y_k, p_k)
        except Exception:
            pr = np.nan

        ece   = ece_binary(p_k, y_k)
        aece  = adaptive_ece_binary(p_k, y_k)
        brier = brier_binary(p_k, y_k)
        nll   = nll_binary(p_k, y_k)
        ae    = auc_err(v_k, p_k, y_k) if variance is not None else np.nan

        # Confidence on the RC curve:
        #   * UQ methods -> -variance (more variance = less confidence)
        #   * baseline   -> Maximum Class Probability
        if variance is not None and np.var(v_k) > 1e-12:
            conf_rc = -v_k
        else:
            conf_rc = mcp_confidence(p_k)
        _, _, aurc = risk_coverage_aurc(conf_rc, p_k, y_k)

        if cal_probs is not None and cal_labels is not None:
            ycal_k, pcal_k = _valid(cal_labels[:, k], cal_probs[:, k])
            if len(ycal_k) > 20 and 0 < ycal_k.sum() < len(ycal_k):
                cov, sz = split_conformal_binary(pcal_k, ycal_k, p_k, y_k)
            else:
                cov, sz = np.nan, np.nan
        else:
            cov, sz = np.nan, np.nan

        rows.append(dict(
            task=k, roc_auc=roc, pr_auc=pr,
            ece=ece, aece=aece, brier=brier, nll=nll,
            auc_err=ae, aurc=aurc,
            conf_cov=cov, conf_size=sz, n=len(y_k),
        ))

    df = pd.DataFrame(rows)
    metric_cols = ["roc_auc", "pr_auc", "ece", "aece", "brier", "nll",
                   "auc_err", "aurc", "conf_cov", "conf_size"]
    agg = {c: float(np.nanmean(df[c])) for c in metric_cols}
    return agg, df

In [ ]:
# =============================================================================
# 14.  Visualisation
# =============================================================================
def _bootstrap_bin_acc(labels_bin: np.ndarray, n_boot: int,
                       rng: np.random.Generator) -> tuple[float, float]:
    """95 % CI of mean(labels_bin) by non-parametric bootstrap."""
    if len(labels_bin) < 2:
        return np.nan, np.nan
    draws = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, len(labels_bin), len(labels_bin))
        draws[b] = labels_bin[idx].mean()
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


def reliability_plot(probs: np.ndarray, labels: np.ndarray,
                     title: str, save_path: str,
                     n_bins: int = None, n_boot: int = None) -> None:
    """Reliability diagram with sample-count histogram + 95 % bootstrap CIs."""
    n_bins = n_bins or CFG.n_bins
    n_boot = n_boot or CFG.n_bootstrap
    y, p = _valid(labels.ravel(), probs.ravel())
    edges   = np.linspace(0, 1, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_ids = np.digitize(p, edges[1:-1])
    rng = np.random.default_rng(0)

    accs, counts, lo, hi, confs = [], [], [], [], []
    for b in range(n_bins):
        m = bin_ids == b
        if m.sum() == 0:
            accs.append(np.nan)
            counts.append(0)
            lo.append(np.nan)
            hi.append(np.nan)
            confs.append(np.nan)
            continue
        accs.append(float(y[m].mean()))
        confs.append(float(p[m].mean()))
        counts.append(int(m.sum()))
        l, h = _bootstrap_bin_acc(y[m], n_boot, rng)
        lo.append(l)
        hi.append(h)

    fig, (ax, axh) = plt.subplots(
        2, 1, figsize=(4.2, 4.8),
        gridspec_kw={"height_ratios": [3, 1]}, sharex=True,
    )
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    ax.plot(confs, accs, "o-", color="#1f77b4", label="Model")
    # Convert lists to arrays so masked NaNs do not break fill_between.
    confs_a = np.asarray(confs, dtype=float)
    lo_a    = np.asarray(lo,    dtype=float)
    hi_a    = np.asarray(hi,    dtype=float)
    valid   = ~np.isnan(confs_a) & ~np.isnan(lo_a) & ~np.isnan(hi_a)
    if valid.any():
        ax.fill_between(confs_a[valid], lo_a[valid], hi_a[valid],
                        color="#1f77b4", alpha=0.25, label="95 % bootstrap")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Fraction of positives")
    ax.set_title(f"{title}\nECE = {ece_binary(p, y):.4f}")
    ax.legend(loc="upper left", fontsize=7)

    axh.bar(centers, counts, width=(edges[1] - edges[0]) * 0.9,
            color="#1f77b4", alpha=0.6)
    axh.set_xlabel("Predicted probability")
    axh.set_ylabel("Count")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def risk_coverage_plot(confidence: np.ndarray, probs: np.ndarray,
                       labels: np.ndarray, title: str, save_path: str) -> None:
    y, p = _valid(labels.ravel(), probs.ravel())
    c    = confidence.ravel()[~np.isnan(labels.ravel())]
    cov, risk, aurc = risk_coverage_aurc(c, p, y)
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.plot(cov, risk, color="#d62728")
    ax.fill_between(cov, 0, risk, alpha=0.2, color="#d62728")
    ax.set_xlabel("Coverage")
    ax.set_ylabel("Risk (Brier)")
    ax.set_title(f"{title}\nAURC = {aurc:.4f}")
    ax.invert_xaxis()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

In [ ]:
# =============================================================================
# 15.  Single-cell experiment runner
# =============================================================================
def run_single(dataset: str, split: str, arch: str, uq: str, seed: int,
               data_list, smiles, Y) -> tuple[dict, pd.DataFrame]:
    """Train + evaluate ONE configuration cell of the grid."""
    set_seed(seed)
    idx_tr, idx_va, idx_te = SPLIT_FN[split](smiles, seed)
    tr_loader, va_loader, te_loader = make_loaders(
        data_list, idx_tr, idx_va, idx_te,
    )
    num_tasks = Y.shape[1]

    # ---- training ----------------------------------------------------------
    if uq == "ensemble":
        members = []
        for m in range(CFG.M_ensemble):
            set_seed(seed * 1000 + m)
            model = build_model(arch, num_tasks)
            model = train_one_model(model, tr_loader, va_loader,
                                    seed=seed * 1000 + m)
            members.append(model)
    else:
        model = build_model(arch, num_tasks)
        model = train_one_model(model, tr_loader, va_loader, seed=seed)

    # ---- prediction (calibration set = val split, used by conformal) -------
    if uq == "baseline":
        p_cal, y_cal = predict_deterministic(model, va_loader)
        p_te,  y_te  = predict_deterministic(model, te_loader)
        var_te = np.zeros_like(p_te)
    elif uq == "mcdropout":
        p_cal, _, y_cal = predict_mcdropout(model, va_loader, T=CFG.T_mc)
        p_te, var_te, y_te = predict_mcdropout(model, te_loader, T=CFG.T_mc)
    elif uq == "ensemble":
        p_cal, _, y_cal = predict_ensemble(members, va_loader)
        p_te, var_te, y_te = predict_ensemble(members, te_loader)
    else:
        raise ValueError(f"Unknown uq method: {uq}")

    # ---- metrics -----------------------------------------------------------
    agg, per_task = per_task_metrics(
        p_te,
        var_te if uq != "baseline" else None,
        y_te,
        cal_probs=p_cal, cal_labels=y_cal,
    )
    agg.update(dict(dataset=dataset, split=split, arch=arch, uq=uq, seed=seed))

    per_task.insert(0, "dataset", dataset)
    per_task.insert(1, "split",   split)
    per_task.insert(2, "arch",    arch)
    per_task.insert(3, "uq",      uq)
    per_task.insert(4, "seed",    seed)

    # ---- diagnostic plots (only for seed 0 -- keep figure count manageable)
    if seed == 0:
        tag = f"{dataset}_{split}_{arch}_{uq}_seed{seed}"
        reliability_plot(
            p_te, y_te,
            title=f"{dataset.upper()} / {split} / {arch}-{uq}",
            save_path=os.path.join(CFG.fig_dir, f"rel_{tag}.png"),
        )
        # Match the confidence used in the AURC computation.
        if uq == "baseline":
            conf_rc = mcp_confidence(p_te.ravel())
        else:
            conf_rc = -var_te.ravel()
        risk_coverage_plot(
            conf_rc, p_te, y_te,
            title=f"{dataset.upper()} / {split} / {arch}-{uq}",
            save_path=os.path.join(CFG.fig_dir, f"rc_{tag}.png"),
        )

    _empty_cache()
    return agg, per_task


def aggregate_across_seeds(results: list[dict]) -> pd.DataFrame:
    """Mean +/- Std across SEEDS.  Tasks were already macro-averaged
    inside per_task_metrics."""
    df = pd.DataFrame(results)
    keys = ["dataset", "split", "arch", "uq"]
    metrics = [c for c in df.columns if c not in keys + ["seed"]]
    rows = []
    for key, g in df.groupby(keys):
        row = dict(zip(keys, key))
        for m in metrics:
            row[f"{m}_mean"] = float(np.nanmean(g[m]))
            row[f"{m}_std"]  = float(np.nanstd(g[m], ddof=1)) if len(g) > 1 else 0.0
        row["n_seeds"] = len(g)
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# =============================================================================
# 16.  Checkpointed main()  (point 10)
# =============================================================================
RAW_COLUMNS = [
    "dataset", "split", "arch", "uq", "seed",
    "roc_auc", "pr_auc",
    "ece", "aece", "brier", "nll",
    "auc_err", "aurc",
    "conf_cov", "conf_size",
]
KEY_COLUMNS = ["dataset", "split", "arch", "uq", "seed"]

RAW_CSV      = os.path.join(CFG.out_dir, "raw_per_seed.csv")
PER_TASK_CSV = os.path.join(CFG.out_dir, "per_task_results.csv")
TOX21_CSV    = os.path.join(CFG.out_dir, "tox21_per_task_appendix.csv")
SUMMARY_CSV  = os.path.join(CFG.out_dir, "summary_mean_std_across_seeds.csv")
TABLE_CSV    = os.path.join(CFG.out_dir, "table_main.csv")


def _load_checkpoint() -> tuple[pd.DataFrame, set]:
    if os.path.isfile(RAW_CSV):
        try:
            df_done = pd.read_csv(RAW_CSV)
            missing = [c for c in KEY_COLUMNS if c not in df_done.columns]
            if missing:
                print(f"[checkpoint]  {RAW_CSV} missing columns {missing} -- "
                      f"starting fresh.")
                return pd.DataFrame(columns=RAW_COLUMNS), set()
            done_keys = set(
                tuple(row) for row in
                df_done[KEY_COLUMNS].itertuples(index=False, name=None)
            )
            print(f"[checkpoint]  loaded {len(df_done)} completed cells "
                  f"from {RAW_CSV}")
            return df_done, done_keys
        except Exception as e:
            print(f"[checkpoint]  could not parse {RAW_CSV} ({e}) -- "
                  f"starting fresh.")
    else:
        print(f"[checkpoint]  no existing {RAW_CSV}; starting a new run.")
    return pd.DataFrame(columns=RAW_COLUMNS), set()


def _append_row(agg: dict) -> None:
    row = {c: agg.get(c, np.nan) for c in RAW_COLUMNS}
    df_row = pd.DataFrame([row], columns=RAW_COLUMNS)
    header_needed = not os.path.isfile(RAW_CSV)
    df_row.to_csv(RAW_CSV, mode="a", header=header_needed, index=False)


def _append_per_task(pt: pd.DataFrame) -> None:
    header_needed = not os.path.isfile(PER_TASK_CSV)
    pt.to_csv(PER_TASK_CSV, mode="a", header=header_needed, index=False)


def main() -> None:
    _fetch_inputs()
    df_done, done_keys = _load_checkpoint()

    grid = list(itertools.product(
        CFG.datasets, CFG.splits, CFG.architectures, CFG.uq_methods, CFG.seeds,
    ))
    total     = len(grid)
    remaining = [cell for cell in grid if cell not in done_keys]
    print(f"[plan]  total cells: {total}  |  done: {total - len(remaining)}  "
          f"|  remaining: {len(remaining)}")

    datasets_cache: dict[str, tuple] = {}
    pbar = tqdm(total=len(remaining), desc="experiments", dynamic_ncols=True)
    n_ok, n_fail, n_skip = 0, 0, 0

    for cell in grid:
        ds, split, arch, uq, seed = cell

        if cell in done_keys:
            if n_skip < 5:
                print(f"Skipping {ds}/{split}/{arch}/{uq}/seed{seed}  (done)")
            elif n_skip == 5:
                print("... further already-done cells skipped silently.")
            n_skip += 1
            continue

        if ds not in datasets_cache:
            try:
                datasets_cache[ds] = load_dataset(ds)
            except Exception as e:
                print(f"[FAIL-DATASET]  {ds}: {e}")
                for bad in [k for k in remaining
                            if k[0] == ds and k not in done_keys]:
                    row = dict(zip(KEY_COLUMNS, bad))
                    _append_row(row)
                    done_keys.add(bad)
                    pbar.update(1)
                continue
        data_list, smiles, Y = datasets_cache[ds]

        try:
            agg, pt = run_single(ds, split, arch, uq, seed,
                                 data_list, smiles, Y)
            _append_row(agg)
            _append_per_task(pt)
            done_keys.add(cell)
            n_ok += 1
            pbar.set_postfix(ok=n_ok, fail=n_fail,
                             last=f"{ds}/{split}/{arch}/{uq}/s{seed}")

        except KeyboardInterrupt:
            print("\n[interrupt]  user stopped run. Progress is on disk.")
            pbar.close()
            break

        except Exception as e:
            print(f"[FAIL]  {ds}/{split}/{arch}/{uq}/seed{seed}: "
                  f"{type(e).__name__}: {e}")
            n_fail += 1
            fail_row = {c: np.nan for c in RAW_COLUMNS}
            fail_row.update(dict(dataset=ds, split=split, arch=arch,
                                 uq=uq, seed=seed))
            _append_row(fail_row)
            done_keys.add(cell)
            pbar.set_postfix(ok=n_ok, fail=n_fail)

        finally:
            pbar.update(1)
            _empty_cache()

    pbar.close()
    print(f"\n[run]  this session  -- ok={n_ok}, fail={n_fail}, skip={n_skip}")

    if not os.path.isfile(RAW_CSV):
        print("[final]  no results on disk -- nothing to aggregate.")
        return

    raw = pd.read_csv(RAW_CSV)
    metric_cols = [c for c in RAW_COLUMNS if c not in KEY_COLUMNS]
    raw_valid = raw.dropna(subset=metric_cols, how="all").copy()
    print(f"[final]  rows on disk: {len(raw)}  |  valid: {len(raw_valid)}")

    if os.path.isfile(PER_TASK_CSV):
        per_task_all = pd.read_csv(PER_TASK_CSV)
        tox21_pt = per_task_all[per_task_all.dataset == "tox21"]
        tox21_pt.to_csv(TOX21_CSV, index=False)
        print(f"[final]  wrote {TOX21_CSV} ({len(tox21_pt)} rows)")

    summary = aggregate_across_seeds(raw_valid.to_dict(orient="records"))
    summary.to_csv(SUMMARY_CSV, index=False)

    cols = [
        "dataset", "split", "arch", "uq",
        "roc_auc_mean",  "roc_auc_std",
        "brier_mean",    "brier_std",
        "ece_mean",      "ece_std",
        "aece_mean",     "aece_std",
        "nll_mean",      "nll_std",
        "auc_err_mean",  "auc_err_std",
        "aurc_mean",     "aurc_std",
        "conf_cov_mean", "conf_cov_std",
        "conf_size_mean","conf_size_std",
    ]
    cols = [c for c in cols if c in summary.columns]
    pretty = summary[cols].copy()
    for c in pretty.columns:
        if c.endswith("_mean") or c.endswith("_std"):
            pretty[c] = pretty[c].round(4)
    pretty.to_csv(TABLE_CSV, index=False)

    print("\n==== FINAL SUMMARY (Mean +/- Std across seeds) ====")
    print(pretty.to_string(index=False))
    print(
        f"\nSaved:\n  {RAW_CSV}\n  {PER_TASK_CSV}\n  {TOX21_CSV}\n  "
        f"{SUMMARY_CSV}\n  {TABLE_CSV}"
    )


if __name__ == "__main__":
    main()


[fetch]  tox21.csv.gz  already present
[fetch]  BBBP.csv  already present
[fetch]  contextpred.pth  already present
[checkpoint]  loaded 72 completed cells from ./results/raw_per_seed.csv
[plan]  total cells: 108  |  done: 72  |  remaining: 36


experiments:   0%|          | 0/36 [00:00<?, ?it/s]

Skipping tox21/scaffold/gin_jk/baseline/seed0  (done)
Skipping tox21/scaffold/gin_jk/baseline/seed1  (done)
Skipping tox21/scaffold/gin_jk/baseline/seed2  (done)
Skipping tox21/scaffold/gin_jk/mcdropout/seed0  (done)
Skipping tox21/scaffold/gin_jk/mcdropout/seed1  (done)
... further already-done cells skipped silently.
[tox21]  molecules:  7823  |  tasks: 12  |  label density: 0.829


experiments:  50%|█████     | 18/36 [1:19:01<2:00:31, 401.74s/it, fail=0, last=tox21/mw/dmpnn/ensemble/s2, ok=18]   

[bbbp]  molecules:  2039  |  tasks:  1  |  label density: 1.000


experiments: 100%|██████████| 36/36 [2:08:26<00:00, 214.06s/it, fail=0, last=bbbp/mw/dmpnn/ensemble/s2, ok=36]         


[run]  this session  -- ok=36, fail=0, skip=72
[final]  rows on disk: 108  |  valid: 108
[final]  wrote ./results/tox21_per_task_appendix.csv (936 rows)

==== FINAL SUMMARY (Mean +/- Std across seeds) ====
dataset    split           arch        uq  roc_auc_mean  roc_auc_std  brier_mean  brier_std  ece_mean  ece_std  aece_mean  aece_std  nll_mean  nll_std  auc_err_mean  auc_err_std  aurc_mean  aurc_std  conf_cov_mean  conf_cov_std  conf_size_mean  conf_size_std
   bbbp       mw          dmpnn  baseline        0.8401       0.0129      0.1832     0.0216    0.1106   0.0292     0.1088    0.0204    0.5699   0.0622           NaN          NaN     0.1255    0.0148         0.8585        0.0341          1.2764         0.0590
   bbbp       mw          dmpnn  ensemble        0.8386       0.0049      0.2100     0.0173    0.1377   0.0395     0.1468    0.0188    0.6219   0.0540        0.7483       0.0321     0.1369    0.0014         0.8244        0.0447          1.2992         0.0540
   bbbp       mw